In [1]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import os
import glob
import pyarrow as pa
import pyarrow.parquet as pq
import matplotlib as plt

import matplotlib.pyplot as plt
import seaborn as sns
import textwrap

In [31]:
#file locations
parquet_file_paths={
    "patient": r"Client_Data_files\Parquets\synthetic_patients.parquet",
    "encounter": r"Client_Data_files\Parquets\synthetic_encounters.parquet",
    "hospitals": r"Client_Data_files\Parquets\synthetic_hospitals.parquet",
    "provider": r"Client_Data_files\Parquets\synthetic_providers.parquet",    
}

In [32]:
#reading the parquet files
patient_df = pd.read_parquet(parquet_file_paths['patient'])
encounter_df = pd.read_parquet(parquet_file_paths['encounter'])
hospital_df = pd.read_parquet(parquet_file_paths['hospitals'])
provider_df = pd.read_parquet(parquet_file_paths['provider'])



In [33]:
# Creating a data map for easy access
data_map={
    "patient": patient_df,
    "encounter": encounter_df,
    "hospitals": hospital_df,
    "provider": provider_df
}

In [34]:
for key, df in data_map.items():
    print(f"Dataframe: {key}")
    print(f"Shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")    
    for col in df.columns:
        if df[col].isna().sum() > 0:
            print(f"Column '{col}' has {df[col].isna().sum()} missing values.")
    print("")

Dataframe: patient
Shape: (100000, 16)
Columns: ['patient_id', 'first_name', 'last_name', 'date_of_birth', 'gender', 'race', 'ethnicity', 'primary_language', 'zip_code', 'insurance_type', 'household_income', 'education_level', 'age', 'cultural_background', 'preferred_provider_language', 'cultural_preferences']

Dataframe: encounter
Shape: (200000, 17)
Columns: ['encounter_id', 'patient_id', 'provider_id', 'encounter_date', 'encounter_type', 'primary_diagnosis', 'length_of_stay', 'total_cost', 'cultural_background', 'primary_language', 'languages_spoken', 'cultural_competency_rating', 'cultural_match_score', 'language_match', 'patient_satisfaction', 'treatment_adherence', 'return_visit_30_days']

Dataframe: hospitals
Shape: (200, 14)
Columns: ['hospital_id', 'hospital_name', 'hospital_type', 'zip_code', 'bed_count', 'teaching_hospital', 'trauma_center', 'language_services_available', 'cultural_competency_program', 'interpreter_services_24_7', 'community_health_programs', 'overall_rating

### Deriving Race and Ethnicity for doctors and correcting race for patients

In [35]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import subprocess
import sys

# Install ethnicolr library for race prediction
try:
    from ethnicolr import census_ln
except ImportError:
    print("Installing ethnicolr...")
    # Note: ethnicolr requires TensorFlow. This installation might take a moment.
    subprocess.run([sys.executable, "-m", "pip", "install", "ethnicolr"], check=True)
    from ethnicolr import census_ln

In [36]:
print(patient_df['race'].value_counts())
print("")
print(patient_df['ethnicity'].value_counts())

race
White                        65181
Hispanic or Latino           12997
Black or African American    12011
Asian                         5932
Other                         1943
Native American               1936
Name: count, dtype: int64

ethnicity
Not Hispanic or Latino    84859
Hispanic or Latino        15141
Name: count, dtype: int64


In [37]:
patient_df_temp=patient_df[patient_df['race']=='Hispanic or Latino'][['patient_id','first_name','last_name','race','ethnicity']].copy()
patient_df_temp.head()

,patient_id,first_name,last_name,race,ethnicity
6,PAT_000007,Ming,Smith,Hispanic or Latino,Not Hispanic or Latino
15,PAT_000016,Li,Miller,Hispanic or Latino,Hispanic or Latino
20,PAT_000021,Fatima,Brown,Hispanic or Latino,Hispanic or Latino
29,PAT_000030,Ahmed,Wang,Hispanic or Latino,Not Hispanic or Latino
31,PAT_000032,William,Thomas,Hispanic or Latino,Not Hispanic or Latino


In [38]:
patient_race_pred=census_ln(patient_df_temp, 'last_name')
patient_race_pred.head()

2025-09-27 20:09:54,446 - INFO - Preserving 12965 duplicate rows based on column 'last_name'
2025-09-27 20:09:54,447 - INFO - Data filtering summary: 12997 → 12997 rows (kept 100.0%)
2025-09-27 20:09:54,453 - INFO - Merging demographic data for 12997 records...
2025-09-27 20:09:54,525 - INFO - Matched 12997 of 12997 rows (100.0%)
2025-09-27 20:09:54,526 - INFO - Added columns: pct2prace, pctaian, pctapi, pctblack, pcthispanic, pctwhite


,patient_id,first_name,last_name,race,ethnicity,pctwhite,pctblack,pctapi,pctaian,pct2prace,pcthispanic
0,PAT_000007,Ming,Smith,Hispanic or Latino,Not Hispanic or Latino,73.35,22.22,0.40,0.85,1.63,1.56
1,PAT_000016,Li,Miller,Hispanic or Latino,Hispanic or Latino,85.81,10.41,0.42,0.63,1.31,1.43
2,PAT_000021,Fatima,Brown,Hispanic or Latino,Hispanic or Latino,60.71,34.54,0.41,0.83,1.86,1.64
3,PAT_000030,Ahmed,Wang,Hispanic or Latino,Not Hispanic or Latino,3.25,0.19,94.47,0.03,1.73,0.33
4,PAT_000032,William,Thomas,Hispanic or Latino,Not Hispanic or Latino,55.53,38.17,1.63,1.01,2.00,1.66


In [39]:
race_mapping={
    'white': 'White',
    'black': 'Black or African American',
    'api': 'Asian',    
    'aian': 'Native American',
    '2prace': 'Other'
}

In [40]:
race_cols=['pctwhite','pctblack','pctapi','pctaian','pct2prace']
patient_race_pred['derived_race'] = patient_race_pred[race_cols].idxmax(axis=1).str.replace('pct', '').map(race_mapping)
patient_race_pred.head()

,patient_id,first_name,last_name,race,ethnicity,pctwhite,pctblack,pctapi,pctaian,pct2prace,pcthispanic,derived_race
0,PAT_000007,Ming,Smith,Hispanic or Latino,Not Hispanic or Latino,73.35,22.22,0.40,0.85,1.63,1.56,White
1,PAT_000016,Li,Miller,Hispanic or Latino,Hispanic or Latino,85.81,10.41,0.42,0.63,1.31,1.43,White
2,PAT_000021,Fatima,Brown,Hispanic or Latino,Hispanic or Latino,60.71,34.54,0.41,0.83,1.86,1.64,White
3,PAT_000030,Ahmed,Wang,Hispanic or Latino,Not Hispanic or Latino,3.25,0.19,94.47,0.03,1.73,0.33,Asian
4,PAT_000032,William,Thomas,Hispanic or Latino,Not Hispanic or Latino,55.53,38.17,1.63,1.01,2.00,1.66,White


In [41]:
patient_df[patient_df['race']=='Hispanic or Latino'].shape

(12997, 16)

In [42]:
patient_df[patient_df['race']=='Hispanic or Latino'].head()

,patient_id,first_name,last_name,date_of_birth,gender,race,ethnicity,primary_language,zip_code,insurance_type,household_income,education_level,age,cultural_background,preferred_provider_language,cultural_preferences
6,PAT_000007,Ming,Smith,1975-08-13 19:28:50.538981,F,Hispanic or Latino,Not Hispanic or Latino,Spanish,18210,Medicare,50335,High School,50,Other/Mixed,Spanish,Culturally Similar Provider; Same Language Pro...
15,PAT_000016,Li,Miller,1944-09-03 19:28:50.538993,M,Hispanic or Latino,Hispanic or Latino,Vietnamese,75629,Private,46488,Graduate,81,Hispanic/Latino,Vietnamese,Culturally Similar Provider; Same Language Pro...
20,PAT_000021,Fatima,Brown,1994-10-14 19:28:50.539000,M,Hispanic or Latino,Hispanic or Latino,English,25917,Medicare,105776,Bachelor's,30,Hispanic/Latino,English,Culturally Similar Provider
29,PAT_000030,Ahmed,Wang,1938-12-27 19:28:50.539012,M,Hispanic or Latino,Not Hispanic or Latino,English,69620,Uninsured,18641,Some College,86,Other/Mixed,English,Culturally Similar Provider
31,PAT_000032,William,Thomas,1990-09-02 19:28:50.539015,F,Hispanic or Latino,Not Hispanic or Latino,Spanish,35624,Medicare,44268,Bachelor's,35,Other/Mixed,Spanish,Culturally Similar Provider; Same Language Pro...


In [43]:
# Create a mapping from patient_id to derived_race
id_to_derived_race = dict(zip(patient_race_pred['patient_id'], patient_race_pred['derived_race']))


# Update the race column only for Hispanic or Latino patients
patient_df.loc[patient_df['race'] == 'Hispanic or Latino', 'race'] = \
    patient_df.loc[patient_df['race'] == 'Hispanic or Latino', 'patient_id'].map(id_to_derived_race)

#### Deriving Provider race and etnicity

In [44]:
provider_race_predictions = census_ln(provider_df, 'last_name')

2025-09-27 20:10:13,049 - INFO - Preserving 4968 duplicate rows based on column 'last_name'
2025-09-27 20:10:13,050 - INFO - Data filtering summary: 5000 → 5000 rows (kept 100.0%)
2025-09-27 20:10:13,052 - INFO - Merging demographic data for 5000 records...
2025-09-27 20:10:13,140 - INFO - Matched 5000 of 5000 rows (100.0%)
2025-09-27 20:10:13,145 - INFO - Added columns: pct2prace, pctaian, pctapi, pctblack, pcthispanic, pctwhite


In [45]:
# Derive race for the provider_df as it is missing from the source data
print("Deriving race for providers from last names...")
race_cols = ['pctwhite','pctblack','pctapi','pctaian','pct2prace']
provider_df['provider_race'] = provider_race_predictions[race_cols].idxmax(axis=1).str.replace('pct', '').map(race_mapping)
print("Provider race derivation complete.")

Deriving race for providers from last names...
Provider race derivation complete.


In [47]:
# Deriving provider ethnicity from the race_predictions
print("Deriving ethnicity for providers from race predictions...")
provider_df['provider_ethnicity'] = race_predictions['pcthispanic'].apply(lambda x: 'Hispanic or Latino' if float(x) >= 50 else 'Not Hispanic or Latino')
print("Provider ethnicity derivation complete.")


Deriving ethnicity for providers from race predictions...
Provider ethnicity derivation complete.


In [51]:
provider_df.columns
provider_df.head()

,provider_id,npi_number,first_name,last_name,specialty,practice_zip_code,years_experience,medical_school_country,board_certified,languages_spoken,...,cultural_certifications,minority_health_experience,community_involvement,patient_satisfaction_score,communication_rating,cultural_competency_rating,hospital_affiliation,accepts_new_patients,provider_race,provider_ethnicity
0,PROV_00001,7639158633,Luis,Johnson,Oncology,81627,6,United States,True,English,...,None,False,None,4.075961,4.571673,3.35,HOSP_110,True,White,Not Hispanic or Latino
1,PROV_00002,1794461566,Sofia,Perez,Psychiatry,77312,5,United States,False,English; Arabic,...,NCQA Cultural Competency Recognition,False,None,4.293616,4.162912,4.75,HOSP_176,True,White,Hispanic or Latino
2,PROV_00003,9322011489,Sofia,Hassan,Internal Medicine,44303,31,United States,False,English,...,None,True,None,3.332870,4.460706,3.34,HOSP_164,False,White,Not Hispanic or Latino
3,PROV_00004,8899037876,Robert,Hassan,Emergency Medicine,13105,8,United States,True,English,...,None,True,None,3.665661,5.000000,3.32,HOSP_102,True,White,Not Hispanic or Latino
4,PROV_00005,7406879593,Robert,Yang,Pediatrics,51213,19,United States,True,English,...,Language Services Certification,False,None,3.330108,5.000000,4.02,HOSP_136,True,Asian,Not Hispanic or Latino


### Placeholder

In [52]:
for key, df in data_map.items():
    print(f"Dataframe: {key}")
    print(f"Shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")    
    for col in df.columns:
        if df[col].isna().sum() > 0:
            print(f"Column '{col}' has {df[col].isna().sum()} missing values.")
    print("")

Dataframe: patient
Shape: (100000, 16)
Columns: ['patient_id', 'first_name', 'last_name', 'date_of_birth', 'gender', 'race', 'ethnicity', 'primary_language', 'zip_code', 'insurance_type', 'household_income', 'education_level', 'age', 'cultural_background', 'preferred_provider_language', 'cultural_preferences']

Dataframe: encounter
Shape: (200000, 17)
Columns: ['encounter_id', 'patient_id', 'provider_id', 'encounter_date', 'encounter_type', 'primary_diagnosis', 'length_of_stay', 'total_cost', 'cultural_background', 'primary_language', 'languages_spoken', 'cultural_competency_rating', 'cultural_match_score', 'language_match', 'patient_satisfaction', 'treatment_adherence', 'return_visit_30_days']

Dataframe: hospitals
Shape: (200, 14)
Columns: ['hospital_id', 'hospital_name', 'hospital_type', 'zip_code', 'bed_count', 'teaching_hospital', 'trauma_center', 'language_services_available', 'cultural_competency_program', 'interpreter_services_24_7', 'community_health_programs', 'overall_rating

In [62]:
# Merge all data into a single master DataFrame for training
master_df = pd.merge(encounter_df, patient_df, on='patient_id',suffixes=('', '_pat'))
master_df = pd.merge(master_df, provider_df, on='provider_id',suffixes=('', '_prov'))
master_df = pd.merge(master_df, hospital_df, left_on='hospital_affiliation', right_on='hospital_id',suffixes=('', '_hosp'))

In [63]:
master_df.to_csv('Client_Data_files/master_df.csv', index=False)

In [59]:
master_df.shape

(200000, 66)